# Evaluation - Anonymization & Pseudonymization 

This notebook evaluates **exclusively**:
- **Anonymization** (for `ANON_LABELS`)
- **Pseudonymization** (for `PSEUDO_LABELS`)

For each experiment (Model × Original/Processed) and task, it calculates:
- **TP / FP / FN**
- **Precision / Recall / F1**

**Outputs:**
- Export: **CSV** with error examples (Leak / missing mapping)
- Export: **CSV** with **False-Positive examples** (Over-Scrubbing)

> **Note:** False Positives (FP) are counted here as "additional replacements in the output that do not correspond to any gold label".
> To achieve this, tokens in the format `[LABEL]` or `[LABEL_n]` are extracted from the output and matched against gold spans.

## 1) Setup & Configuration
4 combinations: Llama/Mistral × Original/Processed

In [ ]:
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
EXPERIMENTS = {
    "Llama_Original": {
        "results": "data/output_original/grascco_llama_combined.json",
        "mapping": "data/output_original/mapping_llama.json",
        "use_gold_text_field": "text_original"  
    },
    "Llama_Processed": {
        "results": "data/output_processed/grascco_llama_combined.json",
        "mapping": "data/output_processed/mapping_llama.json",
        "use_gold_text_field": "text_original"
    },
    "Mistral_Original": {
        "results": "data/output_original/grascco_mistral_combined.json",
        "mapping": "data/output_original/mapping_mistral.json",
        "use_gold_text_field": "text_original"
    },
    "Mistral_Processed": {
        "results": "data/output_processed/grascco_mistral_combined.json",
        "mapping": "data/output_processed/mapping_mistral.json",
        "use_gold_text_field": "text_original"
    },
}

ANON_LABELS = {
    "CONTACT_PHONE", "CONTACT_EMAIL", "ID", "DATE", "DATE_BIRTH",
    "LOCATION_STREET", "LOCATION_ZIP"
}

PSEUDO_LABELS = {
    "NAME_PATIENT", "NAME_DOCTOR", "NAME_RELATIVE",
    "LOCATION_CITY", "LOCATION_HOSPITAL", "LOCATION_COUNTRY", "NAME_TITLE"
}

ALL_LABELS = ANON_LABELS | PSEUDO_LABELS

# Token pattern in Output: [LABEL] oder [LABEL_1]
REPLACEMENT_PATTERN = re.compile(r"\[(?:[A-Z_]+)(?:_\d+)?\]")


## 2) Helper Functions
- **Load JSON**: Utilities to load data.
- **Normalization**: Robust leakage check (whitespace/case normalization).
- **Token Extraction**: Extract tokens from output to identify false positives.

In [ ]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def normalize(text: str) -> str:
    if not text:
        return ""
    # normalize whitespace + casefold for robust matching 
    text = text.casefold()
    text = re.sub(r"\s+", " ", text).strip()
    return text

def extract_predicted_tokens(text: str):
    if not text:
        return []
    return REPLACEMENT_PATTERN.findall(text)

def gold_allowed_prefixes(entry, gold_label_set=ALL_LABELS):
    """Erlaubte Token-Prefixe je Dokument aus Gold-Spans.
    Wir erlauben sowohl [LABEL] als auch [LABEL_...] (Prefix mit Unterstrich).
    """
    prefixes = set()
    for sp in entry.get("spans", []):
        label = sp.get("label")
        if label in gold_label_set:
            prefixes.add(f"[{label}]")   # anon exact token
            prefixes.add(f"[{label}_")   # pseudo prefix
    return prefixes


## 3) Span-based Evaluation (TP/FN) + Output-based FP
This section defines the logic for matching gold spans against model output and detecting false positives.

In [ ]:
def get_gold_text(entry, cfg):
    field = cfg.get("use_gold_text_field", "text_original")
    txt = entry.get(field) or entry.get("text_original") or entry.get("text") or ""
    return txt

def span_gold_value(gold_text, span):
    st, en = int(span.get("start")), int(span.get("end"))
    return gold_text[st:en]

# Normalize titles for more tolerant matching
TITLE_PREFIXES = [
    "frau", "herr", "fr", "hr",
    "dr", "dr.", "prof", "prof.", "pd", "pd.",
    "dr. med.", "dr med", "dr. med"
]

def strip_common_prefixes(s: str) -> str:
    """
    Remove common German title/salutation prefixes to avoid false mismatches:
    e.g., 'Frau Example Person' vs 'Example Person'
    """
    s = normalize(s)
    # Remove repeated prefixes at the beginning
    changed = True
    while changed:
        changed = False
        for pref in TITLE_PREFIXES:
            if s.startswith(pref + " "):
                s = s[len(pref)+1:].strip()
                changed = True
                break
    return s

def mapping_matches_gold(meta_original: str, gold: str) -> bool:
    """
    Tolerant match:
    - Exact match after normalization
    - Or substring match (gold in original or original in gold)
    - Additionally try matches after stripping common prefixes/titles.
    """
    o = normalize(str(meta_original or ""))
    g = normalize(str(gold or ""))
    if not o or not g:
        return False

    if o == g:
        return True

    # Substring match (handles 'Frau Example Person' vs 'Example Person')
    if g in o or o in g:
        return True

    # Title-stripped variants
    o2 = strip_common_prefixes(o)
    g2 = strip_common_prefixes(g)
    if o2 == g2:
        return True
    if g2 in o2 or o2 in g2:
        return True

    return False

def eval_doc_anon_pseudo(entry, output_text, mapping, experiment_name, cfg):
    gold_text = get_gold_text(entry, cfg)
    out_norm = normalize(output_text)
    rows = []

    # TP/FN analysis via Gold Spans
    for sp in entry.get("spans", []):
        label = sp.get("label")
        if label not in ALL_LABELS:
            continue

        gold = span_gold_value(gold_text, sp)
        gold_norm = normalize(gold)

        # Snippet for error analysis
        st, en = int(sp.get("start")), int(sp.get("end"))
        snippet = output_text[max(0, st-60): min(len(output_text), en+60)]

        if label in ANON_LABELS:
            ok = (gold_norm not in out_norm)  # Leak -> False
            rows.append({
                "Experiment": experiment_name,
                "Task": "Anonymization",
                "Doc_ID": entry.get("doc_id", "unknown"),
                "Label": label,
                "Gold": gold,
                "Output_Snippet": snippet,
                "OK": ok,
            })

        if label in PSEUDO_LABELS:
            # Leakage check: Gold must not appear in output
            leaked = (gold_norm in out_norm)

            mapped = False
            mapped_token = None
            if isinstance(mapping, dict):
                for token, meta in mapping.items():
                    if not isinstance(meta, dict):
                        continue
                    if meta.get("label") != label:
                        continue
                    # ✅ Tolerant mapping match
                    if mapping_matches_gold(meta.get("original", ""), gold):
                        mapped = True
                        mapped_token = token
                        break

            ok = (mapped and not leaked)
            rows.append({
                "Experiment": experiment_name,
                "Task": "Pseudonymization",
                "Doc_ID": entry.get("doc_id", "unknown"),
                "Label": label,
                "Gold": gold,
                "Output_Snippet": snippet,
                "OK": ok,
                "Mapped_Token": mapped_token,
                "Leaked": leaked
            })

    # --- FP: Output tokens that do not correspond to any Gold Label ---
    fps = []
    prefixes = gold_allowed_prefixes(entry, ALL_LABELS)
    predicted_tokens = extract_predicted_tokens(output_text)

    for tok in predicted_tokens:
        if not any(tok.startswith(pref) for pref in prefixes):
            fps.append(tok)

    return rows, fps

## 4) Execution: Load & Evaluate All Experiments

In [ ]:
all_rows = []
fp_rows = []  # Examples for FPs (Over-scrubbing)

for exp_name, cfg in EXPERIMENTS.items():
    print("Loading", exp_name)
    data = load_json(cfg["results"])
    mapping = load_json(cfg["mapping"]) if cfg.get("mapping") else {}

    for entry in data:
        out_text = entry.get("text_deidentified", "")
        if isinstance(out_text, list):
            out_text = " ".join(map(str, out_text))

        rows, fps = eval_doc_anon_pseudo(entry, out_text, mapping, exp_name, cfg)
        all_rows.extend(rows)

        if fps:
            fp_rows.extend([
                {
                    "Experiment": exp_name,
                    "Doc_ID": entry.get("doc_id", "unknown"),
                    "False_Positive_Token": tok,
                    "Output_Preview": out_text[:250]
                } for tok in fps
            ])

df = pd.DataFrame(all_rows)
df_fp = pd.DataFrame(fp_rows)

print("Rows (gold-span eval):", len(df))
print("FP tokens:", len(df_fp))
df.head()


## 5) Metrics: TP / FP / FN + Precision / Recall / F1

In [ ]:
def compute_metrics(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0.0
    return precision, recall, f1

def token_label(tok):
    inner = tok.strip("[]")
    return inner.split("_")[0] if "_" in inner else inner

summary_rows = []

for exp in sorted(df["Experiment"].unique()):
    exp_fp = df_fp[df_fp["Experiment"] == exp].copy()
    if not exp_fp.empty:
        exp_fp["TokLabel"] = exp_fp["False_Positive_Token"].apply(token_label)

    for task in ["Anonymization", "Pseudonymization"]:
        sub = df[(df["Experiment"] == exp) & (df["Task"] == task)]
        tp = int((sub["OK"] == True).sum())
        fn = int((sub["OK"] == False).sum())

        if exp_fp.empty:
            fp = 0
        else:
            # Task-specific FP assignment:
            # - Labels in the respective set count as FP for this task
            # - Unknown labels (not in ALL_LABELS) count as FP for both tasks (Over-scrubbing/hallucination)
            if task == "Anonymization":
                fp = int((exp_fp["TokLabel"].isin(ANON_LABELS) | (~exp_fp["TokLabel"].isin(ALL_LABELS))).sum())
            else:
                fp = int((exp_fp["TokLabel"].isin(PSEUDO_LABELS) | (~exp_fp["TokLabel"].isin(ALL_LABELS))).sum())

        precision, recall, f1 = compute_metrics(tp, fp, fn)

        summary_rows.append({
            "Experiment": exp,
            "Task": task,
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "TP": tp,
            "FP": fp,
            "FN": fn
        })

summary = pd.DataFrame(summary_rows).sort_values(["Experiment", "Task"])
summary

## 6) Plots: Precision / Recall / F1 + Counts (TP/FP/FN)

In [ ]:
def plot_metric_bars(summary_df, metric):
    pivot = summary_df.pivot(index="Experiment", columns="Task", values=metric).fillna(0)
    experiments = pivot.index.tolist()
    tasks = pivot.columns.tolist()

    x = np.arange(len(experiments))
    width = 0.8 / max(1, len(tasks))

    plt.figure(figsize=(14, 5))
    for i, t in enumerate(tasks):
        plt.bar(x + i*width, pivot[t].values, width, label=t)

    plt.xticks(x + width*(len(tasks)-1)/2, experiments, rotation=25, ha="right")
    plt.ylim(0, 1.05)
    plt.ylabel(metric)
    plt.title(f"{metric} pro Experiment (Anon vs Pseudo)")
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_metric_bars(summary, "Precision")
plot_metric_bars(summary, "Recall")
plot_metric_bars(summary, "F1")


In [ ]:
def plot_counts(summary_df, task):
    s = summary_df[summary_df["Task"] == task].copy().sort_values("Experiment")
    x = np.arange(len(s))
    width = 0.25

    plt.figure(figsize=(14, 5))
    plt.bar(x - width, s["TP"].values, width, label="TP")
    plt.bar(x,         s["FP"].values, width, label="FP")
    plt.bar(x + width, s["FN"].values, width, label="FN")

    plt.xticks(x, s["Experiment"].tolist(), rotation=25, ha="right")
    plt.ylabel("Count")
    plt.title(f"Counts pro Experiment – {task}")
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_counts(summary, "Anonymization")
plot_counts(summary, "Pseudonymization")


## 7) Export Results
Saves the evaluation results to CSV and Excel files:
1. **`evaluation_errors_anon_pseudo_with_fp.csv`**: Contains False Negatives (FN) and leakage errors.
2. **`false_positives_over_scrubbing.csv`**: Contains False Positive (FP) tokens found in the output.
3. **`evaluation_report.xlsx`**: A combined Excel report containing the Summary, Errors, and FPs on separate sheets.

In [ ]:
# A) FN / Error cases from Gold Spans
errors = df[df["OK"] == False].copy()
errors.to_csv("evaluation_errors_anon_pseudo_with_fp.csv", index=False)

# B) FP Examples
df_fp.to_csv("false_positives_over_scrubbing.csv", index=False)

# C) Excel Report (3 Sheets)
with pd.ExcelWriter("evaluation_report.xlsx", engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="Summary", index=False)
    errors.to_excel(writer, sheet_name="Errors_FN", index=False)
    df_fp.to_excel(writer, sheet_name="FalsePositives_FP", index=False)

print(" Saved files:")
print(" - evaluation_errors_anon_pseudo_with_fp.csv")
print(" - false_positives_over_scrubbing.csv")
print(" - evaluation_report.xlsx")